In [12]:
import math
import pathlib
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

# Find the repo root, so `from jabarti_llm...` works wherever this is opened.
ROOT = pathlib.Path.cwd()
while not (ROOT / "jabarti_llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

torch.manual_seed(0)

print("torch     :", torch.__version__)
print("repo root :", ROOT)

torch     : 2.13.0+cpu
repo root : r:\Developing\IT\LLM-Projects\jabarti-llm-from-scratch


In [ ]:
from jabarti_llm.config import ModelConfig

TOKENS = ["the", "cat", "sat", "on", "mat"]

cfg = ModelConfig(
    vocab_size=32,
    d_model=8,
    n_heads=2,
    n_layers=2,
    max_seq_len=16,
    dropout=0.1,
)

B = 1
T = len(TOKENS)

x = torch.randn(B, T, cfg.d_model)

print("d_model    :", cfg.d_model)
print("n_heads    :", cfg.n_heads, " -> d_k =", cfg.d_k)
print("n_layers   :", cfg.n_layers)
print("d_ff       :", cfg.d_ff, " <- __post_init__ filled this in: 4 * d_model")
print("vocab_size :", cfg.vocab_size)
print("max_seq_len:", cfg.max_seq_len)

d_model    : 8
n_heads    : 2  -> d_k = 4
n_layers   : 2
d_ff       : 3072  <- __post_init__ filled this in: 4 * d_model
vocab_size : 32
max_seq_len: 16


---

# Part 1 -- What we already have, and what is missing

Two modules are finished and we will not rebuild them:

| module | built in | what it does |
|---|---|---|
| `MultiHeadAttention` | `attention.py`, previous notebook | mixes information **across tokens** |
| `FeedForward` | `feedforward.py` | computes **per token**, after the mixing |

Both have the same signature in and out: `(B, T, d_model) -> (B, T, d_model)`.

So the obvious thing to do is just call one after the other.

In [6]:
from jabarti_llm.model.attention import MultiHeadAttention
from jabarti_llm.model.feedforward import FeedForward

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

print("x     :", tuple(x.shape))
print("attn  :", tuple(attn_out.shape))
print("ffn   :", tuple(y.shape))
print()

print("same shape in and out?", tuple(y.shape) == tuple(x.shape))

x     : (1, 5, 8)
attn  : (1, 5, 8)
ffn   : (1, 5, 8)

same shape in and out? True


---
## The residual: give the gradient a way home

```
Residual = "keep the original, add the change on top."
```



Training works by pushing a gradient backwards through every layer. Each layer
**multiplies** the gradient by something on the way past.

Multiply enough numbers smaller than 1 together and you get zero. That is not a
metaphor -- `float32` really does reach exactly `0.0`. Let us watch it happen.

The experiment: stack `depth` feed-forward sublayers, ask for the gradient at
the **input**, and print how much of it survived the trip.

---

Example
```
gradient norm arriving back at the input

  depth     plain stack   with residual
      1       8.151e-03       2.827e+00
      2       2.503e-05       2.824e+00
      5       3.859e-13       2.818e+00
     10       0.000e+00       2.834e+00
     20       0.000e+00       2.822e+00
     30       0.000e+00       2.824e+00
```

## How ?

It is one line of calculus. For a sublayer `f`:

```
plain:      y = f(x)          ->   dy/dx = f'(x)
residual:   y = x + f(x)      ->   dy/dx = 1 + f'(x)
                                        ^
                                        this
```

Stacking multiplies these together. In the plain stack you multiply `f'` by `f'`
by `f'`... and if each one is around 0.5, thirty of them is `0.5^30`, which is
about one in a billion.

In the residual stack every factor is **`1 + something`**. The `1` is an
unbroken path from the loss straight back to the input -- the *highway*. The
sublayer's contribution is a *detour* off that highway. Even if every `f'` is
zero, the gradient still arrives intact through the `1`s.

---

The residual fixed the backward pass. It made the **forward** pass worse.

Look at what the stack now computes:

```
x1 = x0 + f1(x0)
x2 = x1 + f2(x1)
x3 = x2 + f3(x2)
...

```

---

Nothing ever gets smaller. Every layer *adds*. If a sublayer's output tends to be
a bit larger than its input, layer 2 receives something bigger than layer 1 did,
so it outputs something bigger still, and the growth compounds.

**LayerNorm** is the reset. For each token vector separately it subtracts the
mean and divides by the standard deviation, so whatever comes in, what comes out
has mean 0 and std 1.


In [8]:
ln = nn.LayerNorm(cfg.d_model)

wild_input = torch.tensor([
    [
        [50.0, 52.0, 48.0, 51.0, 49.0, 53.0, 47.0, 50.0]
    ]
])

normed_input = ln(wild_input)

print(wild_input.shape)
print(normed_input.shape)


torch.Size([1, 1, 8])
torch.Size([1, 1, 8])


In [11]:
print("wild_input: ")
print("  mean:", round(wild_input.mean().item(), 3),
      "  std:", round(wild_input.std(unbiased=False).item(), 3))
print()

print("normed_input: ")
print("  mean:", round(normed_input.mean().item(), 3),
      "  std:", round(normed_input.std(unbiased=False).item(), 3))
print()

wild_input: 
  mean: 50.0   std: 1.871

normed_input: 
  mean: 0.0   std: 1.0



---
## -- Pre-norm vs post-norm: *where* the norm goes

We now have a residual and a LayerNorm. There are two ways to combine them, and
the choice is not cosmetic.

```
post-norm  (the 2017 original)     x = LayerNorm( x + Attention(x) )
pre-norm   (GPT-2, and this repo)  x = x + Attention( LayerNorm(x) )
```


# Part 2 - GPT


`gpt.py` is where components become a **model**. Everything before it produced
vectors; this file produces a prediction for the next token and a loss to train on.

It adds four things to the block we just built:

1. the **embedding** at the front (token IDs to vectors) -- built in `embeddings.py`;
2. a **stack** of `n_layers` blocks;
3. `ln_final` and `lm_head` at the back (vectors to a score per vocabulary word);
4. the **loss**.

Start with the stack, because it contains the single most common PyTorch bug a
beginner can write.

In [ ]:
from jabarti_llm.model.attention import MultiHeadAttention
from jabarti_llm.model.feedforward import FeedForward
from jabarti_llm.model.block import TransformerBlock

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

blocks = nn.ModuleList(
    TransformerBlock(cfg)
    for ix in range(cfg.n_layers)
)

In [ ]:
y.shape # (B, T, d_model)

torch.Size([1, 5, 8])

---

##  From vectors to predictions: `ln_final` and `lm_head`

The stack gives us `(B, T, d_model)`. We need `(B, T, vocab_size)`: for every
position, a score for every word that could come next.

Two layers do it.

**`ln_final`.** With pre-norm, every block normalises
on the way *in*, so the last block's output has never been normalised on the way
*out*. `lm_head` would be reading whatever scale 12 layers of additions happened
to produce. One LayerNorm fixes it.

**`lm_head`.** A single `nn.Linear(d_model, vocab_size, bias=False)`. Row *i* of
its weight is a direction; the score for token *i* is how much the hidden vector
points that way.

In [21]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

In [37]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

last = logits[0, -1]
probs = F.softmax(last, dim=-1)

top = torch.topk(probs, 5)

---

## Weight tying: one matrix, two jobs

Look at the two big tables we now have:

```
token_embedding.weight   (vocab_size, d_model)   row i = the vector FOR token i
lm_head.weight           (vocab_size, d_model)   row i = the direction that SCORES token i
```

Same shape. And they are asking the same question from two directions.

* Embedding lookup is `one_hot(i) @ W` -- pull out row *i*.
* The head is `hidden @ W.T` -- dot the hidden vector against every row.

In [ ]:
from jabarti_llm.model.embeddings import InputEmbedding

embedding = InputEmbedding(cfg)
head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

print("token_embedding.weight:", embedding.token_embedding.weight.shape)
print("head.weight:", head.weight.shape)

head.weight = embedding.token_embedding.weight

token_embedding.weight: torch.Size([32, 8])
head.weight: torch.Size([32, 8])


---
## Initialisation

`GPT.__init__` ends with two lines that are easy to skim past:

```python
self.apply(self._init_weights)
self._scale_residual_projections()
```

Both are GPT-2's, both exist for a measurable reason.

---

`normal(0, 0.02)` everywhere

PyTorch's default for `nn.Linear` is a uniform distribution whose width depends
on **fan-in** -- the number of inputs. That means a layer's initial scale changes
with `d_model`, so a 768-wide model starts out with noticeably larger activations
than GPT-2's training recipe assumes. GPT-2 uses one fixed `std=0.02` for
everything instead.

In [ ]:
def _init_weights(module):
    """GPT-2's initialisation: normal(0, 0.02), zeroed biases.

        PyTorch's Linear default depends on fan-in, which leaves a 768-wide
        model's activations larger than GPT-2 assumes and makes the early
        steps of training less stable.
    """

    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weights, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weights, mean=0.0, std=0.02)

def _scale_residual_projections(blocks, config):

    scale = math.sqrt( 2 *  config.n_layers)
    for block in blocks:
        with torch.no_grad():
            block.attn.W_o.weight.div_(scale)
            block.ffn.fc2.weight.div_(scale)


---

## The loss

We have `(B, T, vocab_size)` logits. Training needs one number.

## The targets are the inputs, shifted by one

The model reads position `t` and must predict position `t+1`. So the labels are
just the input sequence moved left by one -- no human annotation anywhere. That
is what "self-supervised" means, and it is why a language model can train on raw
text.

In [46]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7]])

inputs = sentence[:, 0:-1]
targets = sentence[:, 1:]

print("sentence:", sentence.tolist()[0])
print("inputs  :", inputs.tolist()[0])
print("targets :", targets.tolist()[0])

print()
print(f"{'position':>9}{'reads':>8}{'must predict':>14}")
for t in range(inputs.size(1)):
    print(f"{t:>9}{inputs[0, t].item():>8}{targets[0, t].item():>14}")

sentence: [5, 9, 14, 3, 21, 7]
inputs  : [5, 9, 14, 3, 21]
targets : [9, 14, 3, 21, 7]

 position   reads  must predict
        0       5             9
        1       9            14
        2      14             3
        3       3            21
        4      21             7


In [ ]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

print("logits          :", tuple(logits.shape)) # (B, T, vocab_size) 
print("targets          :", tuple(targets.shape)) # (B, T)

# flatten (B*T, vocab_size) and (B*T,) for loss computation

flatten_logits = logits.reshape(-1 , logits.size(-1))
print("flatten_logits          :", tuple(flatten_logits.shape))

flatten_targets = targets.reshape(-1)
print("flatten_targets          :", tuple(flatten_targets.shape))

# === Example:

# logits       (B, T, V)   = (2, 5, 50257)   ← scores for every token at every position
# targets      (B, T)      = (2, 5)          ← the correct token id at each position

# logits.reshape(-1, lg.size(-1))  →  (10, 50257)   ← 10 separate predictions
# targets.reshape(-1)              →  (10,)         ← 10 correct answers

# ===

loss = F.cross_entropy( flatten_logits, flatten_targets )

# loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))


logits          : (1, 5, 32)
targets          : (1, 5)
flatten_logits          : (5, 32)
flatten_targets          : (5,)


---

### `ignore_index`: positions that must not be graded

Batches are rectangular, sentences are not, so short sentences get padded.
Grading the model on predicting `[PAD]` would teach it to produce padding.

`ignore_index=pad_id` drops those positions entirely -- no loss, **and no
gradient**. `ch12` reuses exactly this to drop the *prompt* from grading during
instruction finetuning: write `pad_id` into the targets you do not want scored,
and the same line handles it.

In [ ]:
# fill to 12
sentence = torch.tensor([[5, 9, 14, 3, 21, 7, cfg.pad_id, cfg.pad_id, cfg.pad_id,
                                              cfg.pad_id, cfg.pad_id, cfg.pad_id]])

loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                       ignore_index=cfg.pad_id)